# SMS SPAM DETECTION (NAIVE BAYES)

# Data Collection
- Data is collection from Datasets section on Kaggle

# Data Loading
- Faced Error while loading "'utf-8' codec can't decode bytes in position 606-607: invalid continuation byte"
- Meaning : Python is trying to read a file as UTF-8, but the file is actually saved in a different encoding
  
- Solution : Specified a different encoding (latin-1)

In [1]:
import pandas as pd
df = pd.read_csv("spam.csv",encoding="latin-1")

# Initial Exploration

- After Exploring data i noticed that dataset has 5 columns and 5572 rows
- Out of 5 , 2 rows are of my concern
- After checking missing values , we have missing values on unnecessary rows which willbe dropped
- i will name columns for my ease as well
- Check imbalance of ham vs spam: IR=747 / 4825 ​ = 6.46 (Moderate imbalance-accuracy maybe misleading) 
- Average Message length = 15.50 (short text , perfect for naive bayes)

In [ ]:
df.rename(columns={'v1':'label'},inplace=True)
df.rename(columns={'v2':'message'},inplace=True)
df = df[['label','message']]

In [3]:
# df.shape
# df.head()
# df.isna().sum()
vc = df['label'].value_counts()
vc_avg = df['label'].value_counts().max() / df['label'].value_counts().min()
print(vc)
print(vc_avg)



label
ham     4825
spam     747
Name: count, dtype: int64
6.459170013386881


In [7]:
 message_length = []

 for text in df['message']:
    word = text.split()
    count= len(word)
    message_length.append(count)

df['message_length']=message_length


df['message_length'].mean()

C:\Users\Adeel Rana\AppData\Local\Temp\ipykernel_6580\3986730454.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['message_length']=message_length


np.float64(15.494436468054559)

# Data Cleaning 
- i have dropped unnecessary columns which also have missing values that can disturb our model
- Renamed columns names for easy understanding of data
- lower case text
- punctuatiion noise
- common words like is,this and
- different form of same words(stemming)
- naive bayes assumes clean word counts
- Tokenization

In [4]:
import warnings
warnings.filterwarnings("ignore")

In [5]:
import re
import nltk


In [6]:
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to C:\Users\Adeel
[nltk_data]     Rana\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to C:\Users\Adeel
[nltk_data]     Rana\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to C:\Users\Adeel
[nltk_data]     Rana\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [7]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

stop_words = set(stopwords.words('english'))

Text Cleaning Function

In [9]:
def clean_text(text):
 #Lowercase
 text = text.lower()

#Remove punctuation and special text

 text = re.sub(r'[^a-z\s]', '', text)

#Tokenization
 
 tokens = word_tokenize(text)

#Remove stop words

 cleaned_token=[]
 for word in tokens:
  if word not in stop_words:
    cleaned_token.append(word)

 #Join token back to sentance 

 cleaned_text = ' '.join(cleaned_token)

 return cleaned_text



In [10]:
df['clean_message'] = df['message'].apply(clean_text)

In [13]:
df['clean_message'].isna().sum()
df = df[df['clean_message'].str.strip() !='']